In [15]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import scipy.io as spio
import matplotlib.lines
import scipy.stats as spst
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from mlxtend.evaluate import permutation_test

def readFromTextFile(inputFile):
    accList = []
    iFile = open(inputFile,'r')
    for line in iFile.readlines():
        if 'Evaluating' in line:
            currString = line.split('- ')[1][:-4]
        if '[' in line and ']' in line and currString in line:
            currArr = line.rstrip().split(': [')[1][:-1]
            lossVal = float(currArr.split(', ')[0])
            accVal = float(currArr.split(', ')[1])
            accList.append(accVal)
    iFile.close()
    return accList

def createChanceArray():
    return np.random.uniform(0.75,0.8,size=(1000,))

def generateColors(arr1,arr2):
    colorArr = []
    for a in range(len(arr1)):
        currVal1 = arr1[a]
        currVal2 = arr2[a]
        if currVal1 < .8 and currVal2 < .8:
            colorArr.append('red')
        else:
            currP1 = permutation_test([currVal1],createChanceArray(),method='approximate',num_rounds=1000,seed=0)
            currP2 = permutation_test([currVal2],createChanceArray(),method='approximate',num_rounds=1000,seed=0)
            if currP1 < 0.05 and currP2 < 0.05:
                colorArr.append('blue')
            elif currP1 < 0.05 and currP2 >= 0.05:
                colorArr.append('cyan')
            else:
                colorArr.append('magenta')
    return colorArr

def createPlot(file1,file2,file3,file4,savFigName):
    markers = ['o','v','^','>','<','*','s','X']
    legElem1 = [Line2D([0],[0],color='white',marker='o',markersize=12,markerfacecolor='r',label='Start, Goal p ≥ 0.05'),
                Line2D([0],[0],color='white',marker='o',markersize=12,markerfacecolor='b',label='Start, Goal p < 0.05'),
                Line2D([0],[0],color='white',marker='o',markersize=12,markerfacecolor='cyan',
                       label='Start p < 0.05, Goal p ≥ 0.05'),
                Line2D([0],[0],color='white',marker='o',markersize=12,markerfacecolor='magenta',
                       label='Start p ≥ 0.05, Goal p < 0.05'),
               ]
    hpcStar = readFromTextFile(file1)
    hpcGoal = readFromTextFile(file2)
    hpcColors = generateColors(hpcStar,hpcGoal)
    pfcStar = readFromTextFile(file3)
    pfcGoal = readFromTextFile(file4)
    pfcColors = generateColors(pfcStar,pfcGoal)
    fig = plt.figure(figsize=(18,18))
    ax1 = fig.add_subplot(3,3,1)
    ax2 = fig.add_subplot(3,3,2)
    ax3 = fig.add_subplot(3,3,3)
    ax1.set_xlim([.6,1.01])
    ax1.set_ylim([.6,1.01])
    ax1.set_xlabel('Start Decoding Accuracy',fontsize=14)
    ax1.set_ylabel('Goal Decoding Accuracy',fontsize=14)
    for a in range(len(hpcStar)):
        ax1.scatter(hpcStar[a],hpcGoal[a],c=hpcColors[a],s=100,marker=markers[a])
    ax1.set_title('HPC Decoding Accuracy',fontsize=16)
    ax2.set_xlim([.6,1.01])
    ax2.set_ylim([.6,1.01])
    ax2.set_xlabel('Start Decoding Accuracy',fontsize=14)
    ax2.set_ylabel('Goal Decoding Accuracy',fontsize=14)
    for a in range(len(pfcStar)):
        ax2.scatter(pfcStar[a],pfcGoal[a],c=pfcColors[a],s=100,marker=markers[a])
    ax2.set_title('mPFC Decoding Accuracy',fontsize=16)    
    ax3.legend(handles=legElem1,fontsize=12,loc='center')
    #plt.show()
    fig.savefig(savFigName,bbox_inches='tight')
    plt.close()
    return

hpcF1 = ''
pfcF1 = ''
hpcF2 = ''
pfcF2 = ''
savFig = ''
figVar = createPlot(hpcF1,hpcF2,pfcF1,pfcF2,savFig)
